# Audio Classification, Speaker Embeddings & Metric Learning

This interactive lab covers:
1. **2D Spectrogram CNNs**: Treating log-Mel time-frequency matrices as single-channel images.
2. **Acoustic Event Classification**: Forward pass, global pooling, and cross-entropy loss.
3. **Speaker Embeddings (d-vectors)**: Extracting unit-hypersphere latent speaker representations.
4. **Triplet Margin Metric Learning**: Pulling positive speaker pairs closer while pushing impostors away.
5. **Biometric Verification Evaluation**: Measuring False Acceptance Rate (FAR) and False Rejection Rate (FRR).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('05-speech-audio/audio-classification/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import torch
import torch.nn.functional as F
from audio_classifier import (
    SpectrogramCNNClassifier,
    SpeakerEmbeddingModel,
    compute_triplet_loss,
    evaluate_verification,
)

print('Audio Classification & Speaker Verification engine loaded successfully!')

## 1. 2D Spectrogram Audio Classifier Architecture

We instantiate a 3-stage 2D CNN that processes log-Mel spectrograms of dimension $(B, 1, 40, 50)$.

In [ ]:
torch.manual_seed(42)
num_classes = 4  # Speech, Music, Siren, Dog Bark
model = SpectrogramCNNClassifier(num_classes=num_classes, base_dim=16, embedding_dim=64)

# Synthetic batch of 3 spectrograms (1 channel, 40 mel bins, 50 time frames)
spectrograms = torch.randn(3, 1, 40, 50)
targets = torch.tensor([0, 1, 3])

out = model(spectrograms, targets=targets)
print(f'Input spectrogram shape:       {spectrograms.shape}')
print(f'Latent audio embedding shape:  {out["embedding"].shape}')
print(f'Output class logits:           {out["logits"].shape}')
print(f'Classification Loss:           {out["loss"].item():.4f}')

## 2. Speaker Verification: Unit Spherical Embeddings

Extracting L2-normalized embeddings for biometric speaker verification.

In [ ]:
speaker_model = SpeakerEmbeddingModel(model)

# Construct triplet: Anchor (Speaker A), Positive (Speaker A utterance 2), Negative (Speaker B)
anchor_spec = torch.randn(1, 1, 40, 50)
positive_spec = anchor_spec + 0.1 * torch.randn(1, 1, 40, 50)  # Same speaker with minor noise
negative_spec = torch.randn(1, 1, 40, 50)                     # Different speaker

emb_a = speaker_model(anchor_spec)
emb_p = speaker_model(positive_spec)
emb_n = speaker_model(negative_spec)

print(f'Speaker embedding vector L2 norm: {torch.norm(emb_a).item():.4f}')
triplet_loss = compute_triplet_loss(emb_a, emb_p, emb_n, margin=0.2)
print(f'Triplet Margin Loss:              {triplet_loss.item():.4f}')

## 3. Biometric Verification Decision & Metric Evaluation

Evaluating cosine similarity scores against a verification threshold to calculate FAR and FRR.

In [ ]:
# Simulated cosine similarity verification scores across 6 pairs
scores = torch.tensor([0.91, 0.84, 0.35, 0.72, 0.18, 0.29])
labels = torch.tensor([1,    1,    1,    0,    0,    0   ])  # 1: Same, 0: Impostor

metrics = evaluate_verification(scores, labels, threshold=0.5)
print(f'Threshold:                      0.50')
print(f'False Acceptance Rate (FAR):    {metrics["FAR"] * 100:.1f}%')
print(f'False Rejection Rate (FRR):     {metrics["FRR"] * 100:.1f}%')
print(f'Overall Verification Accuracy:  {metrics["accuracy"] * 100:.1f}%')